# LeetCode 2059 — Minimum Operations to Convert Number bằng Python

Link đề: https://leetcode.com/problems/minimum-operations-to-convert-number/ · Độ khó: **Medium** · Chủ đề: Array, Breadth-First Search

Skeleton LeetCode:

```python
class Solution:
    def minimumOperations(self, nums: List[int], start: int, goal: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hóa "số phép biến đổi **ít nhất**" thành **đường đi ngắn nhất trên đồ thị không trọng số** ⇒ **BFS**.
2. Nhận ra không gian trạng thái **hữu hạn**: chỉ các giá trị `x ∈ [0, 1000]` mới được biến đổi tiếp ⇒ tối đa `1001` trạng thái, mỗi trạng thái `3·len(nums)` cạnh.
3. Xử lý đúng quy tắc đặc biệt: giá trị **ra ngoài** `[0, 1000]` vẫn hợp lệ (có thể **là** `goal`) nhưng không đi tiếp được.
4. Kiểm tra `goal` **ngay khi sinh** trạng thái (không đợi lấy ra khỏi hàng đợi).
5. Tránh bẫy **chỉ số âm** của Python khi dùng mảng `visited`.

## 2. Đề bài & ví dụ

Cho mảng `nums` (các số **khác nhau**), số nguyên `start` và `goal`. Biến `x` ban đầu bằng `start`. Mỗi phép biến đổi: nếu `0 ≤ x ≤ 1000`, chọn chỉ số `i` bất kỳ và đặt `x` thành một trong `x + nums[i]`, `x − nums[i]`, `x ^ nums[i]` (XOR). Mỗi `nums[i]` dùng bao nhiêu lần tùy ý. Phép biến đổi làm `x` ra ngoài `[0, 1000]` vẫn hợp lệ, nhưng sau đó **không** biến đổi thêm được.

Trả về số phép biến đổi ít nhất để biến `start` thành `goal`, hoặc `−1` nếu không thể.

**Ví dụ 1:** `nums = [2,4,12], start = 2, goal = 12` → `2` (`2 + 12 = 14`, `14 − 2 = 12`).

**Ví dụ 2:** `nums = [3,5,7], start = 0, goal = −4` → `2` (`0 + 3 = 3`, `3 − 7 = −4`; bước cuối ra ngoài đoạn — hợp lệ).

**Ví dụ 3:** `nums = [2,8,16], start = 0, goal = 1` → `−1`.

**Ràng buộc:** `1 ≤ len(nums) ≤ 1000`; `−10⁹ ≤ nums[i], goal ≤ 10⁹`; `0 ≤ start ≤ 1000`; `start ≠ goal`; các `nums[i]` khác nhau.

In [1]:
import random
import time
from collections import deque
from typing import List

examples = [(([2, 4, 12], 2, 12), 2), (([3, 5, 7], 0, -4), 2), (([2, 8, 16], 0, 1), -1)]
for (nums, start, goal), exp in examples:
    print(f"nums={nums}, start={start}, goal={goal} -> kỳ vọng {exp}")

nums=[2, 4, 12], start=2, goal=12 -> kỳ vọng 2
nums=[3, 5, 7], start=0, goal=-4 -> kỳ vọng 2
nums=[2, 8, 16], start=0, goal=1 -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1 BFS cho đường đi ngắn nhất không trọng số

BFS lần lượt duyệt các trạng thái theo **số bước** tăng dần; lần đầu chạm tới một trạng thái là bằng số bước nhỏ nhất. Mỗi trạng thái chỉ cần thăm một lần.

### 3.2 Phép XOR với số âm trong Python

Python dùng số nguyên **bù hai vô hạn**: `x ^ n` với `n` âm cho kết quả âm hợp lệ, ví dụ `5 ^ -3 = -8`. Kết quả có thể rơi ngoài `[0, 1000]`.

In [2]:
for x, n in [(2, 12), (5, -3), (1000, 1023), (0, -1)]:
    print(f"{x} + {n} = {x + n}, {x} - {n} = {x - n}, {x} ^ {n} = {x ^ n}")

2 + 12 = 14, 2 - 12 = -10, 2 ^ 12 = 14
5 + -3 = 2, 5 - -3 = 8, 5 ^ -3 = -8
1000 + 1023 = 2023, 1000 - 1023 = -23, 1000 ^ 1023 = 23
0 + -1 = -1, 0 - -1 = 1, 0 ^ -1 = -1


### 3.3 Không gian trạng thái

Chỉ trạng thái trong `[0, 1000]` mới "mở rộng" được ⇒ BFS thăm tối đa `1001` trạng thái, mỗi trạng thái sinh `3n` láng giềng ⇒ `O(1001 · 3n) ≈ 3·10⁶` phép với `n = 1000`.

### 3.4 Chỉ số âm của list

`visited[-4]` **không lỗi** mà trỏ vào `visited[len − 4]`. Mảng `visited` kích thước 1001 chỉ được dùng cho giá trị **trong** `[0, 1000]`.

In [3]:
visited = [False] * 1001
visited[-4] = True
print("đặt visited[-4] thực ra đánh dấu", [i for i, v in enumerate(visited) if v])

đặt visited[-4] thực ra đánh dấu [997]


## 4. Ý tưởng nền tảng / trực giác

Xem mỗi giá trị `x` là một đỉnh; từ `x ∈ [0, 1000]` có cạnh tới `x + a`, `x − a`, `x ^ a` với mọi `a ∈ nums`. Giá trị ngoài đoạn là **đỉnh cụt** (không có cạnh đi ra). Bài toán: khoảng cách BFS từ `start` tới `goal`.

Vì `goal` có thể ngoài đoạn (và là đỉnh cụt), ta kiểm tra **khi sinh** mỗi láng giềng: nếu bằng `goal` thì trả `bước + 1` ngay. Chỉ đẩy vào hàng đợi các láng giềng **trong** đoạn chưa thăm.

Tính tay **Ví dụ 1** (`nums = [2,4,12]`, `start = 2`, `goal = 12`):

| bước | trạng thái lấy ra | láng giềng (trong đoạn) |
|---|---|---|
| 0 | 2 | `2±2, 2^2, 2±4, 2^4, 2±12, 2^12` ⇒ trong đoạn `{0, 4, 6, 14}`; `−2`, `−10` ra ngoài (không phải goal) |
| 1 | 14 | `14 − 2 = 12` = goal ⇒ đáp án **2** |

In [4]:
nums, start, goal = examples[0][0]
layer1 = sorted({y for a in nums for y in (start + a, start - a, start ^ a)})
print("tầng 1 từ 2:", layer1)
for x in layer1:
    hits = [(x, op, a) for a in nums for op, y in (("+", x + a), ("-", x - a), ("^", x ^ a)) if y == goal]
    if hits: print("từ", x, "chạm goal bằng", hits)

tầng 1 từ 2: [-10, -2, 0, 4, 6, 14]
từ 0 chạm goal bằng [(0, '+', 12), (0, '^', 12)]
từ 14 chạm goal bằng [(14, '-', 2), (14, '^', 2)]


## 5. Cách 1 — Brute Force / baseline: BFS "ngây thơ" (kiểm tra goal khi lấy ra, `set` cho visited)

Đẩy **mọi** láng giềng (kể cả ngoài đoạn) vào hàng đợi, kiểm tra `goal` khi lấy ra, và chỉ mở rộng nếu trong đoạn. Đúng, nhưng hàng đợi chứa rất nhiều trạng thái cụt.

In [5]:
def minops_naive(nums, start, goal):
    q = deque([(start, 0)]); seen = {start}
    while q:
        x, d = q.popleft()
        if x == goal:
            return d
        if not (0 <= x <= 1000):
            continue                                  # đỉnh cụt
        for a in nums:
            for y in (x + a, x - a, x ^ a):
                if y not in seen:
                    seen.add(y); q.append((y, d + 1))
    return -1

for (nums, s, g), exp in examples:
    got = minops_naive(nums, s, g); print(got, "kỳ vọng", exp); assert got == exp

2 kỳ vọng 2
2 kỳ vọng 2
-1 kỳ vọng -1


**Độ phức tạp:** vẫn `O(1001 · 3n)` lần sinh, nhưng `seen` (set) có thể chứa tới `~3·10⁶` giá trị ngoài đoạn ⇒ tốn bộ nhớ và chậm hơn. Dùng làm chuẩn đối chiếu.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (trạng thái hữu hạn):** chỉ `[0, 1000]` được mở rộng ⇒ `visited` là mảng `1001` phần tử.

**Insight 2 (kiểm tra sớm):** kiểm tra `y == goal` ngay khi sinh, trước khi xét `y` có trong đoạn hay không ⇒ không cần đẩy các giá trị ngoài đoạn vào hàng đợi.

**Insight 3 (BFS theo tầng):** xử lý từng tầng để biết số bước mà không lưu `(x, d)`.

**Invariant:** khi xử lý tầng `d`, mọi trạng thái trong tầng có khoảng cách đúng bằng `d`, và `goal` chưa xuất hiện ở tầng `< d + 1` (nếu có đã trả về).

### 6.1 Minh họa: số trạng thái thực sự được thăm

Đếm số giá trị trong `[0, 1000]` được thăm và số lần sinh láng giềng cho vài bộ `nums` ngẫu nhiên — luôn `≤ 1001` trạng thái.

In [6]:
def visited_count(nums, start):
    seen = [False] * 1001; seen[start] = True; q = deque([start]); gen = 0
    while q:
        x = q.popleft()
        for a in nums:
            for y in (x + a, x - a, x ^ a):
                gen += 1
                if 0 <= y <= 1000 and not seen[y]:
                    seen[y] = True; q.append(y)
    return sum(seen), gen

rng = random.Random(61)
for n in (1, 3, 30, 1000):
    nums = rng.sample(range(-2000, 2000), n)
    print(f"n={n:>4}: thăm {visited_count(nums, 0)[0]:>4} trạng thái, sinh {visited_count(nums, 0)[1]:>8} láng giềng")

n=   1: thăm 1001 trạng thái, sinh     3003 láng giềng
n=   3: thăm  126 trạng thái, sinh     1134 láng giềng
n=  30: thăm 1001 trạng thái, sinh    90090 láng giềng


n=1000: thăm 1001 trạng thái, sinh  3003000 láng giềng


## 7. Các cách tiếp theo

### 7.1 Cách 2 — BFS theo tầng, `visited` 1001 phần tử, kiểm tra goal khi sinh (bản nộp)

In [7]:
class Solution:
    def minimumOperations(self, nums: List[int], start: int, goal: int) -> int:
        seen = [False] * 1001
        seen[start] = True
        frontier = [start]
        steps = 0
        while frontier:
            steps += 1
            nxt = []
            for x in frontier:
                for a in nums:
                    for y in (x + a, x - a, x ^ a):
                        if y == goal:
                            return steps
                        if 0 <= y <= 1000 and not seen[y]:
                            seen[y] = True
                            nxt.append(y)
            frontier = nxt
        return -1

print([Solution().minimumOperations(*a) for a, _ in examples])

[2, 2, -1]


**Độ phức tạp Cách 2:** `O(1001 · 3n)` thời gian, `O(1001)` bộ nhớ.

### 7.2 Cách 3 — BFS hai chiều (bidirectional)

Tìm từ `start` xuôi và từ `goal` **ngược**. Phép ngược của `x → x + a` là `y → y − a`, của `x − a` là `y + a`, của `x ^ a` là `y ^ a`; nhưng **tiền thân** phải nằm trong `[0, 1000]` (vì chỉ giá trị trong đoạn mới biến đổi được). Gặp nhau khi một trạng thái xuất hiện ở cả hai phía.

In [8]:
def minops_bidirectional(nums, start, goal):
    fwd = {start: 0}; bwd = {goal: 0}
    f_front, b_front = [start], [goal]
    if goal in fwd: return 0
    while f_front and b_front:
        if len(f_front) <= len(b_front):
            nxt = []
            for x in f_front:
                for a in nums:
                    for y in (x + a, x - a, x ^ a):
                        if y in bwd:
                            return fwd[x] + 1 + bwd[y]
                        if 0 <= y <= 1000 and y not in fwd:
                            fwd[y] = fwd[x] + 1; nxt.append(y)
            f_front = nxt
        else:
            nxt = []
            for y in b_front:
                for a in nums:
                    for x in (y - a, y + a, y ^ a):         # tiền thân x với x op a = y
                        if not (0 <= x <= 1000):
                            continue
                        if x in fwd:
                            return fwd[x] + 1 + bwd[y]
                        if x not in bwd:
                            bwd[x] = bwd[y] + 1; nxt.append(x)
            b_front = nxt
    return -1

print([minops_bidirectional(*a) for a, _ in examples])

[2, 2, -1]


Lưu ý: tiền thân `x` (trong đoạn) của `y` qua XOR là `y ^ a` vì `(y ^ a) ^ a = y`; qua cộng là `y − a`; qua trừ là `y + a`. Với dữ liệu ở đây (không gian 1001), BFS hai chiều không nhanh hơn đáng kể — nhưng là kỹ thuật quan trọng khi không gian lớn.

**Độ phức tạp Cách 3:** cùng bậc `O(1001 · 3n)`, thường thăm ít trạng thái hơn.

### 7.3 Cách 4 — BFS với bộ lọc láng giềng bằng `set` toán học (vector hóa nhẹ)

Với mỗi tầng, sinh láng giềng bằng set comprehension; kiểm tra `goal in` set láng giềng. Ngắn gọn, phù hợp Python.

In [9]:
def minops_sets(nums, start, goal):
    seen = {start}; frontier = {start}; steps = 0
    while frontier:
        steps += 1
        nb = {y for x in frontier for a in nums for y in (x + a, x - a, x ^ a)}
        if goal in nb:
            return steps
        frontier = {y for y in nb if 0 <= y <= 1000} - seen
        seen |= frontier
    return -1

print([minops_sets(*a) for a, _ in examples])

[2, 2, -1]


## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 2** (`nums = [3,5,7]`, `start = 0`, `goal = −4`):

In [10]:
def trace(nums, start, goal, max_steps=5):
    seen = [False] * 1001; seen[start] = True; frontier = [start]; steps = 0
    while frontier and steps < max_steps:
        steps += 1; nxt = []
        for x in frontier:
            for a in nums:
                for op, y in (("+", x + a), ("-", x - a), ("^", x ^ a)):
                    if y == goal:
                        print(f"bước {steps}: {x} {op} {a} = {y} = goal"); return steps
                    if 0 <= y <= 1000 and not seen[y]:
                        seen[y] = True; nxt.append(y)
        print(f"bước {steps}: tầng mới {sorted(nxt)}")
        frontier = nxt
    return -1

assert trace([3, 5, 7], 0, -4) == 2

bước 1: tầng mới [3, 5, 7]
bước 2: 3 - 7 = -4 = goal


| bước | tầng | ghi chú |
|---|---|---|
| 1 | `{3, 5, 7}` | `0 − a` âm (ngoài đoạn, không phải goal); `0 ^ a = a` |
| 2 | từ `3`: `3 − 7 = −4` | chạm goal (ngoài đoạn — vẫn hợp lệ) ⇒ **2** |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — BFS theo tầng.

In [11]:
class Solution:
    def minimumOperations(self, nums: List[int], start: int, goal: int) -> int:
        seen = [False] * 1001
        seen[start] = True
        frontier = [start]
        steps = 0
        while frontier:
            steps += 1
            nxt = []
            for x in frontier:
                for a in nums:
                    for y in (x + a, x - a, x ^ a):
                        if y == goal:
                            return steps
                        if 0 <= y <= 1000 and not seen[y]:
                            seen[y] = True
                            nxt.append(y)
            frontier = nxt
        return -1

for (nums, s, g), exp in examples:
    got = Solution().minimumOperations(nums, s, g); print(got, exp); assert got == exp

2 2
2 2
-1 -1


## 10. Vì sao thuật toán đúng

**Đồ thị.** Đỉnh là số nguyên; cạnh `x → y` tồn tại ⇔ `0 ≤ x ≤ 1000` và `y ∈ {x + a, x − a, x ^ a}` với `a ∈ nums`. Một dãy biến đổi hợp lệ là một đường đi từ `start`; mọi đỉnh trung gian (trừ đỉnh cuối) phải trong đoạn. Số phép = độ dài đường đi.

**BFS.** Tầng `d` gồm đúng các đỉnh **trong đoạn** có khoảng cách `d` (quy nạp: tầng `d + 1` là láng giềng chưa thăm, trong đoạn, của tầng `d`). Đỉnh ngoài đoạn không cần đưa vào tầng vì không có cạnh ra. Mọi đỉnh (kể cả ngoài đoạn) ở khoảng cách `d + 1` là láng giềng của một đỉnh tầng `d` ⇒ kiểm tra `y == goal` khi sinh từ tầng `d` phát hiện `goal` ở khoảng cách nhỏ nhất. Nếu BFS cạn mà không gặp ⇒ không có đường đi ⇒ `−1`. `start ≠ goal` nên đáp án `≥ 1`. ∎

### 10.1 Kiểm chứng thực nghiệm

So bốn cách trên dữ liệu ngẫu nhiên (cả `nums` âm, `goal` ngoài đoạn, và trường hợp không thể).

In [12]:
rng = random.Random(1010)
for _ in range(400):
    n = rng.randint(1, 6); nums = rng.sample(range(-60, 61), n)
    start = rng.randint(0, 1000); goal = rng.choice([rng.randint(-100, 1100), rng.randint(0, 1000), start + rng.choice(nums)])
    if goal == start: continue
    ref = minops_naive(nums, start, goal)
    assert ref == Solution().minimumOperations(nums, start, goal) == minops_bidirectional(nums, start, goal) == minops_sets(nums, start, goal), (nums, start, goal)
print("4 cách khớp nhau trên 400 bộ ngẫu nhiên")

4 cách khớp nhau trên 400 bộ ngẫu nhiên


## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums) ≤ 1000`, `R = 1001`.

| Cách | Time | Space |
|---|---|---|
| 1. BFS ngây thơ (set mọi giá trị) | `O(R · 3n)` | `O(R · 3n)` |
| 2. BFS theo tầng, mảng visited | `O(R · 3n)` | `O(R)` |
| 3. BFS hai chiều | `O(R · 3n)` | `O(R)` |
| 4. BFS bằng set | `O(R · 3n)` | `O(R · 3n)` tạm |

In [13]:
rng = random.Random(2059)
nums = rng.sample(range(-10 ** 9, 10 ** 9), 1000)
nums_small = rng.sample(range(1, 2000), 1000)
for label, ns, g in [("không thể", [2, 4, 6] * 1, 1), ("1000 số lớn", nums, 10 ** 9 + 7), ("1000 số nhỏ", nums_small, 999)]:
    for name, f in [("naive", minops_naive), ("submit", Solution().minimumOperations), ("bidir", minops_bidirectional)]:
        t = time.perf_counter(); r = f(ns, 0, g); print(f"{label:<12} {name:<7}: {r:>3} ({(time.perf_counter() - t) * 1000:.0f} ms)")

không thể    naive  :  -1 (0 ms)
không thể    submit :  -1 (0 ms)
không thể    bidir  :  -1 (0 ms)
1000 số lớn  naive  :  -1 (1 ms)
1000 số lớn  submit :  -1 (0 ms)
1000 số lớn  bidir  :  -1 (0 ms)
1000 số nhỏ  naive  :   1 (23 ms)
1000 số nhỏ  submit :   1 (0 ms)
1000 số nhỏ  bidir  :   1 (0 ms)


## 12. Bộ test

In [14]:
ALL = {"naive": minops_naive, "submit": Solution().minimumOperations, "bidir": minops_bidirectional, "sets": minops_sets}
def run_all(nums, s, g):
    res = {k: f(list(nums), s, g) for k, f in ALL.items()}
    assert len(set(res.values())) == 1, (nums, s, g, res)
    return res["submit"]

for (nums, s, g), exp in examples:
    assert run_all(nums, s, g) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kiểm chéo bằng BFS ngây thơ
edge = [(([1], 0, 1000), 1000), (([1], 1000, 1001), 1), (([1], 0, -1), 1), (([1000], 0, 2000), 2),
        (([-1], 5, 7), 2), (([7], 3, 4), 1), (([2], 1, 0), -1), (([1001], 0, 2002), -1)]
for (nums, s, g), exp in edge:
    assert run_all(nums, s, g) == exp, (nums, s, g, run_all(nums, s, g))
rng = random.Random(20590)
for _ in range(300):
    nums = rng.sample(range(-1200, 1201), rng.randint(1, 8)); s = rng.randint(0, 1000); g = rng.randint(-1500, 2500)
    if g != s: run_all(nums, s, g)
print("Ví dụ, 8 edge case, 300 bộ ngẫu nhiên: OK")

Ví dụ, 8 edge case, 300 bộ ngẫu nhiên: OK


### 12.1 Kiểm tra tính chất

- Nếu đáp án `d ≠ −1` thì có một dãy `d` phép cụ thể (dựng lại bằng mảng cha) cho ra `goal`, mọi giá trị trung gian trong `[0, 1000]`.
- Hoán vị `nums` không đổi đáp án.
- Thêm một số vào `nums` không làm đáp án tăng (nếu đã khả thi).

In [15]:
def path(nums, start, goal):
    par = {start: None}; q = deque([start])
    while q:
        x = q.popleft()
        for a in nums:
            for op, y in (("+", x + a), ("-", x - a), ("^", x ^ a)):
                if y == goal:
                    seq = [(x, op, a, y)]
                    while par[x] is not None:
                        px, pop, pa = par[x]; seq.append((px, pop, pa, x)); x = px
                    return seq[::-1]
                if 0 <= y <= 1000 and y not in par:
                    par[y] = (x, op, a); q.append(y)
    return None

rng = random.Random(121)
f = Solution().minimumOperations
for _ in range(300):
    nums = rng.sample(range(-50, 51), rng.randint(1, 5)); s = rng.randint(0, 1000); g = rng.randint(-100, 1100)
    if g == s: continue
    d = f(nums, s, g); p = path(nums, s, g)
    if d == -1: assert p is None
    else:
        assert len(p) == d and p[-1][3] == g and all(0 <= st[0] <= 1000 for st in p)
        assert all({"+": st[0] + st[2], "-": st[0] - st[2], "^": st[0] ^ st[2]}[st[1]] == st[3] for st in p)
        assert f(nums + [rng.choice([x for x in range(-60, 61) if x not in nums])], s, g) <= d
    sh = nums[:]; rng.shuffle(sh); assert f(sh, s, g) == d
print("Dựng lại đường đi hợp lệ, độ dài = đáp án; hoán vị/thêm số: đúng (300 bộ)")
print("ví dụ đường đi:", path([2, 4, 12], 2, 12), path([3, 5, 7], 0, -4))

Dựng lại đường đi hợp lệ, độ dài = đáp án; hoán vị/thêm số: đúng (300 bộ)
ví dụ đường đi: [(2, '-', 2, 0), (0, '+', 12, 12)] [(0, '+', 3, 3), (3, '-', 7, -4)]


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ chấp nhận `goal` trong `[0, 1000]`

In [16]:
def wrong_goal_in_range(nums, start, goal):
    seen = [False] * 1001; seen[start] = True; fr = [start]; steps = 0
    while fr:
        steps += 1; nxt = []
        for x in fr:
            for a in nums:
                for y in (x + a, x - a, x ^ a):
                    if 0 <= y <= 1000:                          # SAI: lọc trước khi so goal
                        if y == goal: return steps
                        if not seen[y]: seen[y] = True; nxt.append(y)
        fr = nxt
    return -1

for (nums, s, g), exp in examples:
    print("sai :", wrong_goal_in_range(nums, s, g), "| đúng:", exp)

sai : 2 | đúng: 2
sai : -1 | đúng: 2
sai : -1 | đúng: -1


**Vì sao sai:** đề cho phép phép cuối đưa `x` ra ngoài đoạn — Ví dụ 2 có `goal = −4`. **Sửa:** so `goal` **trước** khi lọc theo đoạn.

### Lỗi 2 — Tiếp tục biến đổi từ giá trị ngoài đoạn

In [17]:
def wrong_continue_outside(nums, start, goal, LIM=3000):
    q = deque([(start, 0)]); seen = {start}
    while q:
        x, d = q.popleft()
        if x == goal: return d
        if abs(x) > LIM: continue                               # SAI: cho đi tiếp ngoài [0, 1000]
        for a in nums:
            for y in (x + a, x - a, x ^ a):
                if y not in seen: seen.add(y); q.append((y, d + 1))
    return -1

nums, s, g = [600], 500, 1700         # tự dựng (đối chiếu lại đề gốc): 500 -> 1100 (ngoài) -> 1700 KHÔNG được phép
print("sai :", wrong_continue_outside(nums, s, g), "| đúng:", Solution().minimumOperations(nums, s, g), "| naive:", minops_naive(nums, s, g))

sai : 2 | đúng: -1 | naive: -1


**Vì sao sai:** sau khi `x` ra ngoài `[0, 1000]`, **không** được biến đổi thêm. `500 + 600 = 1100` đã ngoài đoạn nên không thể cộng tiếp ra `1700`. **Sửa:** chỉ mở rộng trạng thái trong đoạn.

### Lỗi 3 — Dùng mảng `visited` với chỉ số âm

In [18]:
def wrong_negative_index(nums, start, goal):
    seen = [False] * 1001; seen[start] = True; q = deque([(start, 0)])
    while q:
        x, d = q.popleft()
        if x == goal: return d
        if not (0 <= x <= 1000): continue
        for a in nums:
            for y in (x + a, x - a, x ^ a):
                if -1001 <= y <= 1000 and not seen[y]:            # SAI: seen[-k] là seen[1001-k]
                    seen[y] = True; q.append((y, d + 1))
                elif y == goal:
                    return d + 1
    return -1

nums, s, g = [4, 5], 35, 960              # tự dựng (đối chiếu lại đề gốc), tìm bằng thử ngẫu nhiên
print("sai :", wrong_negative_index(nums, s, g), "| đúng:", Solution().minimumOperations(nums, s, g), "| naive:", minops_naive(nums, s, g))

sai : 186 | đúng: 185 | naive: 185


**Vì sao sai:** các giá trị âm như `−k` đánh dấu nhầm `seen[1001 − k]`; về sau khi BFS chạm giá trị thật đó thì nó bị coi là "đã thăm" ⇒ bỏ lỡ đường đi ngắn nhất (ở đây đáp án bị lệch 1). Python không báo lỗi chỉ số âm. **Sửa:** chỉ dùng `seen[y]` khi `0 ≤ y ≤ 1000`.

### Lỗi 4 — DFS (không cho đường ngắn nhất)

In [19]:
def wrong_dfs(nums, start, goal):
    seen = {start}; st = [(start, 0)]
    while st:
        x, d = st.pop()
        for a in nums:
            for y in (x + a, x - a, x ^ a):
                if y == goal: return d + 1                      # SAI: lần chạm đầu tiên theo DFS
                if 0 <= y <= 1000 and y not in seen:
                    seen.add(y); st.append((y, d + 1))
    return -1

for (nums, s, g), exp in examples + [(([1, 3], 0, 9), Solution().minimumOperations([1, 3], 0, 9))]:
    print("sai :", wrong_dfs(nums, s, g), "| đúng:", exp)

sai : 2 | đúng: 2
sai : 10 | đúng: 2
sai : -1 | đúng: -1
sai : 3 | đúng: 3


**Vì sao sai:** DFS tìm **một** đường đi, không phải đường ngắn nhất. **Sửa:** BFS.

### Lỗi 5 — Đánh dấu `visited` khi **lấy ra** thay vì khi **đẩy vào**

In [20]:
def late_mark(nums, start, goal):
    q = deque([(start, 0)]); seen = [False] * 1001; pushes = 0
    while q:
        x, d = q.popleft()
        if seen[x]: continue
        seen[x] = True                                           # đánh dấu muộn
        for a in nums:
            for y in (x + a, x - a, x ^ a):
                if y == goal: return d + 1, pushes
                if 0 <= y <= 1000 and not seen[y]:
                    q.append((y, d + 1)); pushes += 1
    return -1, pushes

nums = list(range(1, 300)); r, pushes = late_mark(nums, 0, 10 ** 6)
print(f"đáp án {r}, số lần đẩy vào hàng đợi: {pushes} (đánh dấu sớm: ≤ 1001)")

đáp án -1, số lần đẩy vào hàng đợi: 400913 (đánh dấu sớm: ≤ 1001)


**Vì sao không tốt:** kết quả đúng, nhưng một trạng thái có thể bị đẩy vào hàng đợi **rất nhiều lần** trước khi được lấy ra ⇒ hàng đợi phình tới `O(R · 3n)`. **Sửa:** đánh dấu ngay khi đẩy vào.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số phép biến đổi ít nhất" + phép biến đổi có chi phí 1 ⇒ **BFS** trên không gian trạng thái.
- Tìm **giới hạn** của không gian trạng thái trong đề (ở đây `[0, 1000]`).
- Đích có thể là **trạng thái cụt** ⇒ kiểm tra đích khi sinh.
- Python: cẩn thận chỉ số âm, dùng mảng `visited` chỉ cho miền hợp lệ.
- Không gian lớn hai đầu biết trước ⇒ cân nhắc BFS hai chiều.

## 15. Biến thể của bài

1. **752 Open the Lock**, **433 Minimum Genetic Mutation**: BFS trên chuỗi trạng thái.
2. **991 Broken Calculator**: tham lam ngược từ đích.
3. **Chi phí khác nhau cho từng phép** (ví dụ XOR tốn 2) ⇒ Dijkstra / 0-1 BFS (Bài tập 5).
4. **Đếm số cách** đạt `goal` với số bước ít nhất — BFS kèm đếm.
5. **Miền rộng hơn** (`[0, 10⁶]`) ⇒ BFS hai chiều có lợi rõ.

### 15.1 Code biến thể: đếm số dãy biến đổi tối thiểu (BFS kèm đếm)

In [21]:
def count_shortest(nums, start, goal):
    dist = {start: 0}; ways = {start: 1}; frontier = [start]; d = 0; hit = 0
    while frontier and not hit:
        d += 1; nxt = []
        for x in frontier:
            for a in nums:
                for y in (x + a, x - a, x ^ a):
                    if y == goal:
                        hit += ways[x]; continue
                    if 0 <= y <= 1000:
                        if y not in dist:
                            dist[y] = d; ways[y] = ways[x]; nxt.append(y)
                        elif dist[y] == d:
                            ways[y] += ways[x]
        frontier = nxt
    return (d, hit) if hit else (-1, 0)

def count_brute(nums, start, goal, D):
    cnt = 0
    def go(x, k):
        nonlocal cnt
        if k == D:
            cnt += x == goal; return
        if not (0 <= x <= 1000): return
        for a in nums:
            for y in (x + a, x - a, x ^ a): go(y, k + 1)
    go(start, 0); return cnt

rng = random.Random(15)
for _ in range(200):
    nums = rng.sample(range(-8, 9), rng.randint(1, 3)); s = rng.randint(0, 20); g = rng.randint(-10, 30)
    if g == s: continue
    d, w = count_shortest(nums, s, g)
    if d != -1 and d <= 4: assert w == count_brute(nums, s, g, d) and d == Solution().minimumOperations(nums, s, g)
print("Đếm số dãy tối thiểu khớp vét cạn;", count_shortest([2, 4, 12], 2, 12))

Đếm số dãy tối thiểu khớp vét cạn; (2, 8)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS ngây thơ | set mọi giá trị | `O(R·3n)` | lớn | Kiểm chứng |
| 2. BFS theo tầng | mảng 1001, check khi sinh | `O(R·3n)` | `O(R)` | **Bản nộp** |
| 3. BFS hai chiều | gặp nhau ở giữa | `O(R·3n)` | `O(R)` | Không gian lớn |
| 4. BFS bằng set | set comprehension | `O(R·3n)` | tạm lớn | Code ngắn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `([1], 0, 1000)`; (b) `([1], 0, −1)`; (c) `([2], 1, 0)`; (d) `([1000], 0, 2000)`.

**Bài 2 (tính tay):** Với `nums = [3], start = 0, goal = 7`: liệt kê tầng 1, 2, 3 của BFS và đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `neighbors(x, nums)` trả danh sách 3n láng giềng.

**Bài 4 (tự viết lại):** Viết `min_ops_path(nums, start, goal)` trả **dãy phép** cụ thể (danh sách `(x, op, a, y)`) có độ dài nhỏ nhất, hoặc `None`.

**Bài 5 (mini challenge):** Phép XOR tốn **2** đơn vị, cộng/trừ tốn 1. Viết `min_cost(nums, start, goal)` bằng Dijkstra (trạng thái trong `[0, 1000]`, đích có thể ngoài); đối chiếu vét cạn trên dữ liệu nhỏ.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def neighbors(x, nums):
    out = []
    # TODO: với mỗi a trong nums, thêm x + a, x - a, x ^ a
    return out

In [23]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def min_ops_path(nums, start, goal):
    # TODO: BFS lưu cha (x, op, a) của mỗi trạng thái; khi chạm goal thì dựng ngược
    pass

def min_cost(nums, start, goal):
    # TODO: heapq với (chi phí, x); khi sinh y == goal thì cập nhật đáp án ứng viên; chỉ đẩy y trong [0, 1000]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: BFS theo tầng. Mức 2: goal ngoài đoạn vẫn chấp nhận. Mức 3: (a) `1000`; (b) `1`; (c) `−1` (từ số lẻ chỉ tới số lẻ: `1 ± 2`, `1 ^ 2 = 3` đều lẻ); (d) `2` (`0 + 1000 = 1000`, `1000 + 1000 = 2000`).

**Bài 2.** Mức 1: tầng 1: `{3}` (`−3` ngoài đoạn, `0 ^ 3 = 3`). Mức 2: tầng 2: `{6}`; tầng 3: `{5, 9}` (`6 ^ 3 = 5`). Mức 3: tầng 4: `{2, 8, 10, 12}`; bước 5: `10 − 3 = 7` ⇒ đáp án `5`.

**Bài 3.** Mức 1: ba phép. Mức 2: XOR với số âm vẫn hợp lệ. Mức 3: list comprehension.

**Bài 4.** Mức 1: `par[y] = (x, op, a)`. Mức 2: dựng ngược tới `start`. Mức 3: kiểm tra từng bước.

**Bài 5.** Mức 1: Dijkstra vì chi phí không đồng nhất. Mức 2: đích có thể chạm từ nhiều đỉnh với chi phí khác nhau — lấy min. Mức 3: dừng khi chi phí nhỏ nhất trong heap `≥` đáp án hiện tại.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2, 3
f = Solution().minimumOperations
b1 = [f([1], 0, 1000), f([1], 0, -1), f([2], 1, 0), f([1000], 0, 2000)]
print("Bài 1:", b1); assert b1 == [1000, 1, -1, 2]
print("Bài 2:"); r2 = trace([3], 0, 7, max_steps=10); assert r2 == f([3], 0, 7) == 5
def neighbors(x, nums):
    out = []
    for a in nums:
        out += [x + a, x - a, x ^ a]
    return out
assert sorted(neighbors(2, [2, 4, 12])) == sorted([4, 0, 0, 6, -2, 6, 14, -10, 14])
print("Bài 3: OK")

Bài 1: [1000, 1, -1, 2]
Bài 2:
bước 1: tầng mới [3]
bước 2: tầng mới [6]
bước 3: tầng mới [5, 9]
bước 4: tầng mới [2, 8, 10, 12]
bước 5: 10 - 3 = 7 = goal
Bài 3: OK


In [25]:
# Bài 4, 5
import heapq
def min_ops_path(nums, start, goal):
    return path(nums, start, goal)

def min_cost(nums, start, goal):
    INF = float("inf"); dist = {start: 0}; pq = [(0, start)]; best = INF
    while pq:
        c, x = heapq.heappop(pq)
        if c >= best: break
        if c > dist.get(x, INF): continue
        for a in nums:
            for y, w in ((x + a, 1), (x - a, 1), (x ^ a, 2)):
                if y == goal: best = min(best, c + w)
                if 0 <= y <= 1000 and c + w < dist.get(y, INF):
                    dist[y] = c + w; heapq.heappush(pq, (c + w, y))
    return -1 if best == INF else best

def min_cost_brute(nums, start, goal, maxc=6):
    best = INF = float("inf")
    def go(x, c):
        nonlocal best
        if c >= best or c > maxc: return
        if x == goal and c > 0: best = c; return
        if not (0 <= x <= 1000): return
        for a in nums:
            for y, w in ((x + a, 1), (x - a, 1), (x ^ a, 2)): go(y, c + w)
    go(start, 0); return -1 if best == INF else best

rng = random.Random(5)
for _ in range(150):
    nums = rng.sample(range(-6, 7), rng.randint(1, 3)); s = rng.randint(0, 10); g = rng.randint(-5, 15)
    if g == s: continue
    p = min_ops_path(nums, s, g); assert (p is None) == (f(nums, s, g) == -1) and (p is None or len(p) == f(nums, s, g))
    mc = min_cost(nums, s, g)
    if mc != -1 and mc <= 6: assert mc == min_cost_brute(nums, s, g)
print("Bài 4, 5: OK |", min_ops_path([2, 4, 12], 2, 12), min_cost([2, 4, 12], 2, 12))

Bài 4, 5: OK | [(2, '-', 2, 0), (0, '+', 12, 12)] 2


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
seen = [False] * 1001; seen[start] = True
frontier, steps = [start], 0
while frontier:
    steps += 1; nxt = []
    for x in frontier:
        for a in nums:
            for y in (x + a, x - a, x ^ a):
                if y == goal: return steps          # kiểm tra TRƯỚC khi lọc đoạn
                if 0 <= y <= 1000 and not seen[y]:
                    seen[y] = True; nxt.append(y)
    frontier = nxt
return -1
```

**Checklist:**
- [ ] BFS (không DFS)?
- [ ] So `goal` khi sinh, kể cả `goal` ngoài `[0, 1000]`?
- [ ] Không mở rộng trạng thái ngoài đoạn?
- [ ] Không dùng `seen[y]` với `y` âm?
- [ ] Đánh dấu `visited` khi đẩy vào hàng đợi?